# Cybersecurity Thread Analysis

Standalone exploration of validation, indicator extraction, and safe demo analysis.

In [1]:
import re

print('re imported')

re imported


In [2]:
thread = 'Suspicious login from 203.0.113.10. Contact soc@example.com. See evil.example.com.'
print(thread)

Suspicious login from 203.0.113.10. Contact soc@example.com. See evil.example.com.


## 1. Extract indicators locally

In [3]:
def extract_indicators(text: str) -> dict[str, list[str]]:
    """Extract IPs, domains, and emails without contacting them."""

    patterns = {
        'ipv4': r'\b(?:\d{1,3}\.){3}\d{1,3}\b',
        'domain': r'\b(?:[A-Za-z0-9-]+\.)+(?:com|net|org|example)\b',
        'email': r'\b[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}\b',
    }
    return {name: sorted(set(re.findall(pattern, text))) for name, pattern in patterns.items()}

In [4]:
indicators = extract_indicators(thread)
print(indicators)
assert '203.0.113.10' in indicators['ipv4']
assert 'soc@example.com' in indicators['email']

{'ipv4': ['203.0.113.10'], 'domain': ['evil.example.com', 'example.com'], 'email': ['soc@example.com']}


## 2. Map evidence to MITRE ATT&CK

This small local mapper produces candidates only. It does not confirm an incident or contact any indicator.

In [5]:
MITRE_RULES = [
    {
        "id": "T1566",
        "name": "Phishing",
        "tactic": "Initial Access",
        "keywords": ("phishing", "credential harvest"),
    },
    {
        "id": "T1059.001",
        "name": "PowerShell",
        "tactic": "Execution",
        "keywords": ("powershell", "powershell.exe"),
    },
    {
        "id": "T1110",
        "name": "Brute Force",
        "tactic": "Credential Access",
        "keywords": ("password spray", "brute force", "credential stuffing"),
    },
]


def map_mitre_attack(text: str) -> list[dict]:
    """Return ATT&CK candidates supported by explicit keywords."""

    lowered = text.lower()
    return [
        {
            "technique_id": rule["id"],
            "technique": rule["name"],
            "tactic": rule["tactic"],
            "evidence": [keyword for keyword in rule["keywords"] if keyword in lowered],
        }
        for rule in MITRE_RULES
        if any(keyword in lowered for keyword in rule["keywords"])
    ]

In [ ]:
thread_with_behavior = "A phishing email delivered a PowerShell payload and triggered password spray attempts."
attack_candidates = map_mitre_attack(thread_with_behavior)
print(attack_candidates)
assert {item["technique_id"] for item in attack_candidates} == {"T1566", "T1059.001", "T1110"}

## 2. Safe analysis summary

In [6]:
def demo_analysis(indicators: dict[str, list[str]]) -> str:
    """Create a defensive summary without network access."""

    count = sum(len(values) for values in indicators.values())
    return f'Found {count} indicator values. No indicators were contacted or executed.'

In [7]:
print(demo_analysis(indicators))

Found 4 indicator values. No indicators were contacted or executed.
